In [1]:
import sys
import os

# Add the root directory to Python's path so we can import our 'app' modules
sys.path.append(os.path.abspath('..'))

from app.popularity import PopularityRecommender
from app.content_engine import ContentEngine
from app.collaborative_engine import CollaborativeEngine
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
print("1. Loading Popularity Engine...")
pop_engine = PopularityRecommender(data_path="../data/cleaned_products.csv")

print("2. Loading Content Engine (GPU)...")
content_engine = ContentEngine(data_path="../data/cleaned_products.csv")

print("3. Training Collaborative Engine (SVD)...")
cf_engine = CollaborativeEngine()

print("\n--- ALL SYSTEMS READY ---")

1. Loading Popularity Engine...
2. Loading Content Engine (GPU)...
Loading Content Engine...
Vectorizing product text...
Content Engine using device: cuda
3. Training Collaborative Engine (SVD)...
Initializing Collaborative Engine...
Loading data from database...
Training SVD model on 1941 unique user-item interactions...
SVD Model trained successfully.

--- ALL SYSTEMS READY ---


In [3]:
def get_hybrid_recommendations(user_id, product_id_viewed=None):
    """
    Simulates the decision logic:
    1. If user is COLD (no history) -> Show Popularity
    2. If user is WARM (has history) -> Show Collaborative Filtering
    3. If user is viewing a product -> Show Content Similarity
    """
    
    # CASE A: User is viewing a specific product (Context-Aware)
    if product_id_viewed:
        print(f"\n[Context] User is looking at {product_id_viewed}. Showing similar items...")
        recs = content_engine.find_similar_products(product_id_viewed)
        return pd.DataFrame(recs)

    # CASE B: User is on Homepage (Personalization)
    # We predict scores for a sample of items to find the best ones
    print(f"\n[Personalization] Generating homepage for User {user_id}...")
    
    # In a real app, we'd predict for all items. Here, we sample top 100 popular ones to save time.
    candidates = pop_engine.get_recommendations(k=100)
    
    scored_candidates = []
    for item in candidates:
        # Predict how much THIS user would like THIS item
        est_score = cf_engine.predict_score(user_id, item['asin'])
        item['predicted_rating'] = est_score
        scored_candidates.append(item)
    
    # Sort by the SVD predicted score, not popularity!
    recs = sorted(scored_candidates, key=lambda x: x['predicted_rating'], reverse=True)[:5]
    return pd.DataFrame(recs)[['title', 'predicted_rating', 'categoryName']]

# TEST 1: Context (Content-Based)
# Let's find a product ID to "view"
sample_product = pop_engine.df.iloc[0]['asin']
display(get_hybrid_recommendations(user_id="9999", product_id_viewed=sample_product))

# TEST 2: Personalization (Collaborative)
# Let's see what User 1001 (who has history) gets vs User 9999 (who is new)
print("----------------------------------------------------------------")
print("User 1001 Recommendations (Should be personalized):")
display(get_hybrid_recommendations(user_id="1001"))


[Context] User is looking at B001MZV3OO. Showing similar items...


,asin,title,score
0,B01LXICPLA,Catsan Smart Pack Cat Litter 2 Inlays 4kg (PAC...,0.702431
1,B0009X0QSY,"Precious Cat Dr. Elsey's Ultra Cat Litter, 8.1...",0.640255
2,B07BP1RGBF,"Iris Ohyama, Cat litter box, Closed cat toilet...",0.582376
3,B00PH49FVO,"Litter Genie Refill ,blue(4 Pack)",0.577023
4,B003EGIM3O,SureFlap Microchip Cat Flap - White,0.565689


----------------------------------------------------------------
User 1001 Recommendations (Should be personalized):

[Personalization] Generating homepage for User 1001...


,title,predicted_rating,categoryName
0,Numatic Numatic Henry Cleaner Bags - 1 Box (Pa...,1.683237,Vacuums & Floorcare
1,Catsan Hygiene Cat Litter 20L,1.530620,Pet Supplies
2,Andrex Gentle Clean Toilet Rolls - 45 Toilet R...,1.530620,Grocery
3,Amazon Brand – Mama Bear Sensitive Unscented B...,1.530620,Health & Personal Care
4,"Regina Blitz Household Towel, 560 Super-Sized ...",1.530620,Grocery
